# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors (FAIR⁲) Dataset Exploration with `mlcroissant`

This notebook demonstrates how to explore the *FAIR⁲ Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors* dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install -U mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview

Let's review available record sets, fields, columns, and their `@id`s provided by the Croissant metadata structure.

In [ ]:
# List record sets and their fields, referencing only by @id
print("Available Record Sets:")
for record_set in dataset.record_sets:
    print(f"\n- Record Set @id: {record_set['@id']}")
    fields = record_set.get('field', []) or []
    print("  Fields:")
    for field in fields:
        if isinstance(field, dict) and '@id' in field:
            print(f"    - Field @id: {field['@id']}")
        elif isinstance(field, str):
            print(f"    - Field @id: {field}")

## 3. Data Extraction

Extract data from a record set using the record set and field `@id`s from above. 
Here, we will:
- Choose a principal record set (e.g., the main observation table)
- Load all its records into a Pandas DataFrame for further analysis using only its `@id`

In [ ]:
# Let's collect @ids
record_set_ids = [r['@id'] for r in dataset.record_sets]
print("Record Set @ids for extraction:")
print(record_set_ids)

dataframes = {}
for record_set_id in record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df
    print(f"Loaded {len(df)} records from record set {record_set_id}")

# For demonstration, select the first record set (usually the main table)
main_record_set_id = record_set_ids[0] if record_set_ids else None
if main_record_set_id:
    print(f"\nColumns available in record set with @id '{main_record_set_id}':")
    print(dataframes[main_record_set_id].columns.tolist())
    display(dataframes[main_record_set_id].head())

## 4. Exploratory Data Analysis (EDA)

Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and grouping data by attributes. 

All field and record references are done by `@id`.

In [ ]:
# Pick a numeric field by @id for analysis (e.g., age, diagnosis interval; update if actual @id is different)
main_df = dataframes[main_record_set_id]

# List numeric columns by inspecting a sample record (for illustration)
print("Possible numeric fields (@id):", main_df.select_dtypes(include='number').columns.tolist())

# Suppose 'cr:age_at_2nd_primary' is a field @id (replace as needed)
# If no numeric fields exist, you may choose a field that makes sense for this dataset
if len(main_df.select_dtypes(include='number').columns):
    numeric_field_id = main_df.select_dtypes(include='number').columns[0]
else:
    numeric_field_id = main_df.columns[0]  # fallback to first field
print(f"\nUsing numeric field: {numeric_field_id}")

threshold = main_df[numeric_field_id].mean() if pd.api.types.is_numeric_dtype(main_df[numeric_field_id]) else 0

filtered_df = main_df[main_df[numeric_field_id] > threshold]
print(f"Filtered records with {numeric_field_id} > {threshold:.2f}:")
display(filtered_df.head())

# Normalize selected numeric field
filtered_df[f"{numeric_field_id}_normalized"] = (
    filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()
) / filtered_df[numeric_field_id].std()
print(f"Normalized {numeric_field_id} for filtered records:")
display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

# Group by a categorical field (e.g., sex, tumor_site, etc.), referenced by @id
candidate_group_fields = main_df.select_dtypes(include='object').columns.tolist()
group_field_id = candidate_group_fields[0] if candidate_group_fields else None
if group_field_id and group_field_id in filtered_df.columns:
    grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean()
    print(f"\nGrouped data by {group_field_id} (mean {numeric_field_id}):")
    print(grouped_df.head())

## 5. Visualization

Visualize a distribution or relationship between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Distribution of numeric field
plt.figure(figsize=(7,4))
sns.histplot(main_df[numeric_field_id].dropna(), bins=15, kde=True, color='dodgerblue')
plt.title(f'Distribution of {numeric_field_id}')
plt.xlabel(numeric_field_id)
plt.ylabel('Count')
plt.show()

# If group_field_id exists, plot group summary
if group_field_id:
    plt.figure(figsize=(8,5))
    sns.boxplot(x=group_field_id, y=numeric_field_id, data=main_df)
    plt.title(f'{numeric_field_id} by {group_field_id}')
    plt.xticks(rotation=45)
    plt.show()

## 6. Conclusion

In this notebook, we demonstrated how to load, inspect, and analyze a clinical oncology dataset structured with a Croissant schema using the `mlcroissant` library. Data access and analysis was driven entirely by `@id` referencing for record sets and fields, ensuring reproducibility and schema transparency.

- We listed all record sets and fields by their `@id`s
- Loaded the main record set into Pandas for tabular exploration
- Performed simple filtering, normalization, and group statistics using only `@id` notation
- Visualized key field distributions and comparisons

You can extend this notebook by incorporating your own custom EDA, more visualizations, or integrating with ML workflows using the Croissant schema information throughout!